# 03 - Semantic Attribution Analysis

## 项目背景

02告诉我们"哪里有问题"——代码生成retry率高、知识问答dislike率高。但行为数据只能定位问题场景，无法回答"为什么不满意"。本notebook通过语义分析对bad case进行归因分类，找到具体的问题原因。

## 本notebook的目标

对被点踩和被重新生成的对话（bad case），进行问题归因分类：

| 归因类别 | 含义 | 典型表现 |
|---------|------|---------|
| 幻觉 | 模型编造事实 | 虚构人物、伪造论文、错误时间线 |
| 答非所问 | 未理解用户意图 | 用户要代码给了概念解释 |
| 拒绝回答 | 过度安全策略 | "我无法执行""涉及安全性问题" |
| 质量不足 | 方向对但不够好 | 太笼统、缺乏细节、敷衍 |
| 格式不匹配 | 输出形式不符预期 | 用户要代码给了文字说明 |

## 意图分类路径

本项目中 `conversations.csv` 的 `intent` 字段模拟的是生产环境中意图分类模型的输出。实际场景中，用户的query不会自带意图标签，需要通过以下方式获得：

**方式一：关键词规则**
- 根据query中的关键词匹配（如包含"代码""写一个函数"→代码生成，包含"翻译""translate"→翻译）
- 优点：快、可控；缺点：覆盖率有限

**方式二：LLM分类**
- 调用大模型对query做意图分类，prompt示例："请将以下用户问题分类为以下类别之一：代码生成、知识问答、创意写作、翻译、闲聊、数据分析"
- 优点：准确率高、泛化好；缺点：有API成本

**方式三：训练分类模型**
- 用标注数据训练一个轻量文本分类器（如BERT微调）
- 优点：成本低、速度快；缺点：需要标注数据

**方式四：人工标注**
- 将query导出为csv，由标注人员逐条分类
- 优点：准确率最高，可作为其他方式的校准集（golden set）；缺点：成本高、不可规模化

本项目为聚焦归因分析流程，直接使用预分类的intent字段。

## 归因分类：两条路径

拿到bad case后，需要判断每条对话"为什么不好"。本notebook提供两条路径：

### 路径A：关键词规则 + 人工校验
先用关键词规则引擎做粗分类，再由人工逐条检查和纠正。

**第一步：关键词规则粗分类**
- 基于response文本中的关键词特征匹配归因类别
- 例如：出现"抱歉""无法"→拒绝回答；出现虚构人名、伪造年份→幻觉
- 覆盖率有限，大量case会掉进"质量不足"兜底类别——这正是需要人工介入的部分

**第二步：人工校验与纠正**
- 将规则预标注的结果导出为csv，由标注人员逐条检查
- 工作流程：
  1. 阅读用户query，理解用户意图
  2. 阅读模型response，判断回复是否满足意图
  3. 检查规则给出的归因标签是否正确，不正确则修改
  4. 重点检查被兜底到"质量不足"的case，判断是否有更具体的归因
  5. 遇到边界case时记录备注，后续团队讨论对齐标准
- 标注规范：
  - 每条case只选一个主要归因（即使可能有多个问题，选最突出的）
  - 优先选择具体类别（幻觉、拒绝回答），避免滥用"质量不足"兜底
  - 建议至少两人独立标注同一批数据，计算一致率（Cohen's Kappa）
- 优点：准确率高，规则预标注减轻了人工工作量
- 缺点：仍需大量人力，不可规模化

### 路径B：LLM自动打标
- 将query + response拼成prompt，调用大模型做归因分类
- 优点：泛化能力强，能理解语义层面的问题，可规模化
- 缺点：有API成本，需要用人工标注的golden set校验准确率

> 生产环境建议：先用路径A标注几百条建立golden set，再用golden set评估路径B的准确率，达标后全量使用路径B。

## 流程

**Step 1: 筛选Bad Case**
- 从事件表中提取所有dislike和retry事件对应的对话

**Step 2: 归因分类（双路径）**
- 路径A：LLM批量打标
- 路径B：导出csv → 人工标注 → 读回

**Step 3: 归因分布分析**
- 整体归因占比
- 按意图场景交叉分析：每个场景的主要问题是什么

**Step 4: 典型案例展示**
- 每个归因类别展示代表性案例

## 输出

标注后的bad case数据保存至 `output/`，归因分布图表在notebook内展示。

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

# 配色
COLORS = {
    'like': '#2ecc71', 'dislike': '#e74c3c', 'retry': '#f39c12',
    'followup': '#3498db', 'silent': '#95a5a6'
}

INTENT_CN = {
    'code_generation': '代码生成', 'knowledge_qa': '知识问答',
    'creative_writing': '创意写作', 'translation': '翻译',
    'chitchat': '闲聊', 'data_analysis': '数据分析'
}

ATTRIBUTION_COLORS = {
    '幻觉': '#e74c3c',
    '答非所问': '#f39c12',
    '拒绝回答': '#8b5cf6',
    '质量不足': '#3498db',
    '格式不匹配': '#14b8a6'
}

# 加载数据
convs = pd.read_csv('../data/conversations.csv', parse_dates=['timestamp'])
events = pd.read_csv('../data/events.csv', parse_dates=['event_timestamp'])

# 筛选bad case：第一个事件为dislike或retry的对话
first_events = events.sort_values('event_timestamp').groupby('conversation_id').first().reset_index()
bad_ids = first_events[first_events['event_type'].isin(['dislike', 'retry'])]['conversation_id']
bad_cases = convs[convs['conversation_id'].isin(bad_ids)].copy()
bad_cases = bad_cases.merge(first_events[['conversation_id', 'event_type']], on='conversation_id')

print(f'Bad case总数: {len(bad_cases)}')
print(f'\n按触发行为:')
print(bad_cases['event_type'].value_counts())
print(f'\n按意图场景:')
print(bad_cases['intent'].value_counts())

In [ ]:
# === 语义归因：基于规则的预标注 ===

def attribute_cause(row):
    response = row['response']
    query = row['query']
    intent = row['intent']
    
    # 规则1：拒绝回答——response中包含拒绝类关键词
    refuse_keywords = ['抱歉', '无法', '不能', '涉及安全', '建议你自己', '我没有能力']
    if any(kw in response for kw in refuse_keywords):
        return '拒绝回答'
    
    # 规则2：幻觉——response中包含编造类信号
    hallucination_keywords = ['图灵奖', '首次提出', 'Johnson教授', 'Nature上发表', '1985年']
    if any(kw in response for kw in hallucination_keywords):
        return '幻觉'
    
    # 规则3：格式不匹配——用户要代码但response没有代码块，或反过来
    code_intents = ['code_generation', 'data_analysis']
    if intent in code_intents and '```' not in response and '代码' not in response:
        return '格式不匹配'
    
    # 规则4：答非所问——response内容和query的意图明显不匹配
    if intent == 'code_generation' and '思路' in response and '代码' not in response:
        return '答非所问'
    if '不太确定' in response or '具体细节' in response:
        return '答非所问'
    
    # 规则5：以上都不是，归为质量不足
    return '质量不足'

bad_cases['attribution'] = bad_cases.apply(attribute_cause, axis=1)

print('归因分布:')
print(bad_cases['attribution'].value_counts())
print(f'\n归因覆盖率: 100%（所有bad case均已分类）')

In [ ]:
# === 归因分布可视化 ===

# 整体归因占比 - 饼图
attr_counts = bad_cases['attribution'].value_counts()

fig = go.Figure(go.Pie(
    labels=attr_counts.index,
    values=attr_counts.values,
    marker=dict(colors=[ATTRIBUTION_COLORS[a] for a in attr_counts.index]),
    textinfo='label+percent',
    textfont=dict(size=13),
    hole=0.4
))

fig.update_layout(
    title='Bad Case 归因分布',
    template='plotly_white',
    height=450,
    showlegend=False
)

fig.show()

In [ ]:
# === 按意图场景 × 归因类别交叉分析 ===

cross_attr = pd.crosstab(bad_cases['intent'], bad_cases['attribution'], normalize='index') * 100
cross_attr = cross_attr.round(1)
cross_attr = cross_attr.rename(index=INTENT_CN)

# 按不满意率排序（复用02的结论）
intent_order = ['代码生成', '数据分析', '知识问答', '创意写作', '翻译', '闲聊']
cross_attr = cross_attr.reindex(intent_order)

attr_order = ['幻觉', '答非所问', '拒绝回答', '格式不匹配', '质量不足']

fig = go.Figure()

for attr in attr_order:
    if attr in cross_attr.columns:
        fig.add_trace(go.Bar(
            name=attr,
            x=cross_attr.index,
            y=cross_attr[attr],
            marker_color=ATTRIBUTION_COLORS[attr],
            text=[f'{v:.0f}%' for v in cross_attr[attr]],
            textposition='inside',
            textfont=dict(size=10)
        ))

fig.update_layout(
    barmode='stack',
    title='各意图场景 × 归因类别分布',
    xaxis_title='意图场景',
    yaxis_title='百分比 (%)',
    template='plotly_white',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)

fig.show()

# 打印交叉表
print('\n各场景归因占比 (%):')
print(cross_attr[attr_order].to_string())

In [ ]:
# === 路径B：LLM API 自动归因（可选） ===
# 如需使用，请填入你的API key后取消注释运行

"""
import openai

# 填入你的API key
openai.api_key = 'sk-xxx'

ATTRIBUTION_PROMPT = '''你是一个AI产品质量分析师。请对以下对话进行归因分类。

用户问题：{query}
模型回复：{response}

请从以下类别中选择一个最匹配的归因：
- 幻觉：模型编造了不存在的事实、人物、论文、数据
- 答非所问：模型没有理解用户意图，回答方向偏离
- 拒绝回答：模型以安全或能力为由拒绝回答本应回答的问题
- 质量不足：回答方向正确但太笼统、缺乏细节、不够深入
- 格式不匹配：输出形式不符合用户预期（如要代码给了文字）

请只返回类别名称，不要解释。'''

def llm_attribute(row):
    prompt = ATTRIBUTION_PROMPT.format(query=row['query'], response=row['response'])
    resp = openai.ChatCompletion.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    return resp.choices[0].message.content.strip()

# 对前20条做测试
# sample = bad_cases.head(20).copy()
# sample['llm_attribution'] = sample.apply(llm_attribute, axis=1)
# print(sample[['query', 'attribution', 'llm_attribution']].to_string())
"""

print('LLM归因路径已准备，取消注释并填入API key即可运行')
print('建议先用少量样本测试（如前20条），确认prompt效果后再全量运行')

In [ ]:
# === 导出bad case供人工校验 ===

import os
os.makedirs('../output', exist_ok=True)

# 导出：包含规则预标注结果，人工在 attribution_manual 列填写校正后的标签
export = bad_cases[['conversation_id', 'intent', 'event_type', 'query', 'response', 'attribution']].copy()
export = export.rename(columns={'attribution': 'rule_attribution'})
export['attribution_manual'] = ''  # 人工填写列

export.to_csv('../output/bad_cases_for_labeling.csv', index=False, encoding='utf-8-sig')

print(f'已导出 {len(export)} 条bad case至 output/bad_cases_for_labeling.csv')
print(f'字段说明:')
print(f'  rule_attribution  — 关键词规则预标注结果')
print(f'  attribution_manual — 人工校验后的归因标签（待填写）')
print(f'\n标注完成后，将csv放回 output/ 目录，运行下方cell读取结果')

In [ ]:
# === 读取人工标注结果（标注完成后运行此cell） ===

"""
labeled_path = '../output/bad_cases_for_labeling.csv'
labeled = pd.read_csv(labeled_path)

# 检查标注完成度
total = len(labeled)
done = (labeled['attribution_manual'] != '').sum()
print(f'标注进度: {done}/{total} ({done/total*100:.1f}%)')

# 检查标签合法性
valid_labels = {'幻觉', '答非所问', '拒绝回答', '质量不足', '格式不匹配'}
invalid = labeled[
    (labeled['attribution_manual'] != '') & 
    (~labeled['attribution_manual'].isin(valid_labels))
]
if len(invalid) > 0:
    print(f'⚠️ 发现 {len(invalid)} 条非法标签:')
    print(invalid['attribution_manual'].unique())
else:
    print('✅ 所有标签合法')

# 规则 vs 人工一致率
if done > 0:
    compared = labeled[labeled['attribution_manual'] != ''].copy()
    agreement = (compared['rule_attribution'] == compared['attribution_manual']).mean()
    print(f'\n规则 vs 人工一致率: {agreement:.1%}')
"""

print('人工标注完成后，取消上方注释运行即可')

In [ ]:
# === 路径B：LLM API 自动归因（可选） ===

"""
import openai

# 填入你的API key
openai.api_key = 'sk-xxx'

ATTRIBUTION_PROMPT = '''你是一个AI产品质量分析师。请对以下对话进行归因分类。

用户问题：{query}
模型回复：{response}

请从以下类别中选择一个最匹配的归因：
- 幻觉：模型编造了不存在的事实、人物、论文、数据
- 答非所问：模型没有理解用户意图，回答方向偏离
- 拒绝回答：模型以安全或能力为由拒绝回答本应回答的问题
- 质量不足：回答方向正确但太笼统、缺乏细节、不够深入
- 格式不匹配：输出形式不符合用户预期（如要代码给了文字）

请只返回类别名称，不要解释。'''

def llm_attribute(row):
    prompt = ATTRIBUTION_PROMPT.format(query=row['query'], response=row['response'])
    resp = openai.ChatCompletion.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    return resp.choices[0].message.content.strip()

# 先用少量样本测试
# sample = bad_cases.head(20).copy()
# sample['llm_attribution'] = sample.apply(llm_attribute, axis=1)
# print(sample[['query', 'attribution', 'llm_attribution']].to_string())
"""

print('LLM归因路径已准备，取消注释并填入API key即可运行')
print('建议先用20条测试prompt效果，确认后再全量运行')

In [ ]:
# === 各归因类别典型案例（卡片式） ===

from IPython.display import HTML

cards_html = '<div style="font-family: -apple-system, sans-serif;">'
cards_html += '<h3 style="margin-bottom: 20px;">各归因类别典型案例</h3>'

for attr in bad_cases['attribution'].value_counts().index:
    cases = bad_cases[bad_cases['attribution'] == attr]
    sample = cases.sample(1, random_state=42).iloc[0]
    color = ATTRIBUTION_COLORS[attr]
    pct = len(cases) / len(bad_cases) * 100
    intent_cn = INTENT_CN.get(sample['intent'], sample['intent'])
    response_display = sample['response'][:200] + '...' if len(sample['response']) > 200 else sample['response']
    
    cards_html += f'''
    <div style="border-left: 4px solid {color}; background: #f9f9f9; padding: 16px 20px; 
                margin-bottom: 16px; border-radius: 0 8px 8px 0;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
            <span style="background: {color}; color: white; padding: 4px 12px; border-radius: 12px; 
                         font-size: 14px; font-weight: bold;">{attr}</span>
            <span style="color: #888; font-size: 13px;">{len(cases)} 条 · 占比 {pct:.1f}%</span>
        </div>
        <div style="color: #666; font-size: 12px; margin-bottom: 10px;">
            意图场景: {intent_cn} &nbsp;|&nbsp; 触发行为: {sample['event_type']}
        </div>
        <div style="background: white; padding: 12px; border-radius: 6px; margin-bottom: 8px;">
            <div style="color: #333; font-size: 13px; margin-bottom: 4px;">💬 <b>用户</b></div>
            <div style="color: #333; font-size: 13px;">{sample['query']}</div>
        </div>
        <div style="background: white; padding: 12px; border-radius: 6px;">
            <div style="color: #333; font-size: 13px; margin-bottom: 4px;">🤖 <b>模型</b></div>
            <div style="color: #333; font-size: 13px; white-space: pre-wrap;">{response_display}</div>
        </div>
    </div>
    '''

cards_html += '</div>'
HTML(cards_html)

## 关键发现

**1. 归因分布**
- 质量不足占71.2%，是最大的问题类别——但其中大量是关键词规则未能细分的兜底，实际可能包含更多具体原因
- 格式不匹配12.4%，主要集中在数据分析场景（65%）
- 拒绝回答6.8%，创意写作场景占比最高（23%）
- 幻觉6.7%，几乎全部出现在知识问答场景（22%）

**2. 各场景的核心问题不同**
- 代码生成：拒绝回答（12%）+ 格式不匹配（13%），模型倾向于给思路而不是直接给代码
- 数据分析：格式不匹配（65%），模型要求用户提供数据而不是直接给方案
- 知识问答：幻觉（22%）+ 答非所问（10%），事实性错误是主要问题
- 创意写作：拒绝回答（23%），过度安全策略导致不必要的拒绝

**3. 关键词规则的局限性**
- 翻译和闲聊场景100%兜底到质量不足，说明规则完全无法覆盖
- 生产环境中应使用LLM打标（路径B），关键词规则仅作为baseline对照

> 下一步：04 将对追问信号进行拆解，区分"探索式追问"和"纠错式追问"